# 01 · Data Pipeline & Validation
**Brazilian Stock-Bond Correlation Study**

This notebook:
1. Builds (or loads from cache) the master returns dataset
2. Validates each series against known benchmarks
3. Produces a data availability / coverage heatmap
4. Cross-validates synthetic bond returns against ETF proxies (2019+)

> **Run once** — subsequent notebooks load from `data/processed/master_returns.csv`

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns

from fetch import build_master_returns, load_master, CRISES, REGIMES

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

CRISIS_COLORS = {
    "GFC":         "#d62728",
    "Dilma":       "#ff7f0e",
    "Joesley":     "#9467bd",
    "COVID":       "#2ca02c",
    "Americanas":  "#8c564b",
    "Fiscal24":    "#e377c2",
}

ASSET_LABELS = {
    "ibov":      "Ibovespa",
    "ntnb":      "NTN-B 5y",
    "ltn":       "LTN 2y",
    "ntnf":      "NTN-F 10y",
    "lft": "LFT 1y (Tesouro Selic)",
    "ptax":      "BRL/USD",
}

def add_crisis_bands(ax, alpha=0.15):
    """Shade crisis periods on a matplotlib axis."""
    for name, (s, e) in CRISES.items():
        ax.axvspan(pd.Timestamp(s), pd.Timestamp(e),
                   color=CRISIS_COLORS[name], alpha=alpha, label=name)

## 1. Build master dataset

In [ ]:
# Force rebuild = False → uses cache if available; set True to re-fetch
master = build_master_returns(force_rebuild=False)

print(f"Shape         : {master.shape}")
print(f"Date range    : {master.index[0].date()} → {master.index[-1].date()}")
print(f"\nReturn columns: {[c for c in master.columns if c in ASSET_LABELS]}")
print(f"Level columns : {['embi','cdi_level','selic','ipca','brl_usd','sov_oas','yld_diff']}")
print(f"\nFirst 3 rows:")
master.head(3)

## 2. Data coverage heatmap

Check which series have data on each day — important for understanding sample sizes per analysis.

In [ ]:
ret_cols = ["ibov", "ntnb", "ltn", "ntnf", "lft"]

# Monthly availability matrix (1 = data present, 0 = NaN)
avail = master[ret_cols].resample("ME").apply(lambda x: x.notna().mean())
avail.columns = [ASSET_LABELS[c] for c in avail.columns]

fig, ax = plt.subplots(figsize=(13, 4))
sns.heatmap(
    avail.T,
    cmap="YlGn", vmin=0, vmax=1,
    ax=ax, cbar_kws={"label": "% of days with data"},
    linewidths=0,
)
ax.set_title("Data availability by month and asset class", fontsize=13, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")

# Mark crisis periods
for name, (s, e) in CRISES.items():
    s_idx = avail.index.searchsorted(pd.Timestamp(s))
    e_idx = avail.index.searchsorted(pd.Timestamp(e))
    ax.axvspan(s_idx, e_idx, color=CRISIS_COLORS[name], alpha=0.25)

plt.tight_layout()
plt.savefig("../outputs/fig_data_coverage.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_data_coverage.png")

## 3. Price-level chart: cumulative growth of all asset classes

This is the key context chart — it shows each asset's trajectory across all macro regimes.

In [ ]:
# Rebuild cumulative return indices (base = 100 on first common date)
ret_cols = ["ibov", "ntnb", "ltn", "ntnf", "lft"]
sub = master[ret_cols].dropna(how="all")

# Start all series from the first date where ALL return cols have data
common_start = sub.dropna().index[0]
sub = sub[sub.index >= common_start].copy()

# Cumulative log return → price index
price_idx = np.exp(sub.cumsum()) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                          gridspec_kw={"height_ratios": [2, 1]})

# ── Top: cumulative price indices ─────────────────────────────────────────────
ax = axes[0]
colors = ["#1f77b4", "#d62728", "#ff7f0e", "#2ca02c", "#9467bd"]
for i, col in enumerate(ret_cols):
    ax.plot(price_idx.index, price_idx[col], label=ASSET_LABELS[col],
            lw=1.5, color=colors[i])
add_crisis_bands(ax)
ax.set_yscale("log")
ax.set_ylabel("Cumulative return index\n(log scale, base=100)", fontsize=10)
ax.set_title("Brazilian asset classes: cumulative total return (2005–2026)", fontsize=13)

# Add regime labels at top
for name, (s, e) in REGIMES.items():
    mid = pd.Timestamp(s) + (pd.Timestamp(e) - pd.Timestamp(s)) / 2
    if mid >= common_start:
        ax.text(mid, ax.get_ylim()[1] * 0.95, name,
                ha="center", va="top", fontsize=7.5, color="gray",
                rotation=0)

handles_assets = [plt.Line2D([0],[0], color=colors[i], lw=2,
                              label=ASSET_LABELS[col])
                  for i, col in enumerate(ret_cols)]
handles_crisis = [plt.Rectangle((0,0),1,1,
                                  fc=CRISIS_COLORS[n], alpha=0.4, label=n)
                  for n in CRISES]
ax.legend(handles=handles_assets + handles_crisis,
          loc="upper left", fontsize=8.5, ncol=2)

# ── Bottom: Ibovespa drawdown ──────────────────────────────────────────────────
ax2 = axes[1]
ibov_idx = price_idx["ibov"]
drawdown  = (ibov_idx / ibov_idx.cummax() - 1) * 100
ax2.fill_between(drawdown.index, drawdown, 0,
                 color="#1f77b4", alpha=0.4, label="Ibovespa drawdown")
add_crisis_bands(ax2, alpha=0.2)
ax2.set_ylabel("Drawdown (%)", fontsize=10)
ax2.set_xlabel("")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator(2))

plt.tight_layout()
plt.savefig("../outputs/fig_cumulative_returns.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/fig_cumulative_returns.png")

## 4. Validate BCB macro series

Sanity check: EMBI should spike during crises, Selic should match known COPOM cycles.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# EMBI
ax = axes[0]
ax.plot(master.index, master["embi"], color="#d62728", lw=1.2)  # ends Jul 2024
add_crisis_bands(ax)
ax.set_ylabel("EMBI+ Brazil (bps)", fontsize=10)
ax.set_title("Sovereign risk proxy (EMBI+)", fontsize=11)
ax.axhline(y=4.0, color="gray", ls="--", lw=0.8, alpha=0.6)

# Selic
ax = axes[1]
ax.plot(master.index, master["selic"], color="#1f77b4", lw=1.5, label="Selic target")
ax.plot(master.index, master["cdi_level"], color="#ff7f0e", lw=1, ls="--",
        alpha=0.7, label="CDI")
add_crisis_bands(ax)
ax.set_ylabel("Rate (% p.a.)", fontsize=10)
ax.set_title("Selic target rate and CDI", fontsize=11)
ax.legend(fontsize=9)

# BRL/USD
ax = axes[2]
ax.plot(master.index, master["brl_usd"], color="#2ca02c", lw=1.2)
add_crisis_bands(ax)
ax.set_ylabel("BRL / USD", fontsize=10)
ax.set_title("Exchange rate (PTAX)", fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))

plt.suptitle("BCB macro series validation", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/fig_macro_validation.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Validate the constant-maturity construction

Two independent checks that the bond series are built correctly:

1. **LFT vs CDI.** The 1-year Tesouro Selic total return is built from observed PU,
   entirely independently of the CDI series. It must track compounded CDI closely —
   if it does not, the PU roll or the return calculation is wrong.
2. **Roll and coupon diagnostics.** Rolls (a change in the selected maturity) and
   coupon payment dates are the two places a constant-maturity construction can
   inject a spurious return. Neither should show up as an outlier.

In [ ]:
# ── 1. LFT (observed PU) vs CDI (independent BCB series) ──────────────────────
chk = master[["lft", "cdi_ret"]].dropna()
lft_cum = np.exp(chk["lft"].cumsum())
cdi_cum = np.exp(chk["cdi_ret"].cumsum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(lft_cum.index, lft_cum, label="LFT 1y (Tesouro Selic PU)", lw=1.8, color="#2ca02c")
ax.plot(cdi_cum.index, cdi_cum, label="Compounded CDI (BCB SGS 12)", lw=1.4,
        color="#1f77b4", ls="--")
add_crisis_bands(ax)
ax.set_yscale("log")
ax.set_ylabel("Growth of 1 unit (log scale)")
ax.set_title("LFT total return vs compounded CDI", fontsize=11)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

# ── 2. The LFT desagio: where the two series come apart ───────────────────────
ax = axes[1]
gap = (lft_cum / cdi_cum - 1) * 100
ax.plot(gap.index, gap, lw=1.4, color="#d62728")
add_crisis_bands(ax)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_ylabel("LFT cumulative return minus CDI (%)")
ax.set_title("Tracking gap — widens when LFTs trade at a desagio", fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.savefig("../outputs/fig_lft_cdi_crossval.png", dpi=150, bbox_inches="tight")
plt.show()

rho  = chk["lft"].corr(chk["cdi_ret"])
drift = (chk["lft"].mean() - chk["cdi_ret"].mean()) * 252 * 100
print(f"Correlation LFT vs CDI daily returns : {rho:.4f}")
print(f"Annualised return gap (LFT - CDI)    : {drift:+.2f} pp")
print(f"LFT cumulative x{lft_cum.iloc[-1]:.2f}  |  CDI cumulative x{cdi_cum.iloc[-1]:.2f}")
print("\n(rho > 0.9 and |gap| < 1pp confirm the PU-based construction is sound)")

In [ ]:
# ── Roll and coupon diagnostics ───────────────────────────────────────────────
# A roll (change of the selected maturity) or a coupon payment must not show up
# as an outlier: if it does, the same-bond return or the coupon add-back is wrong.
print("=== Mean |daily return| on roll days vs other days ===")
for col in ["ntnb", "ltn", "ntnf", "lft"]:
    roll_col = f"{col}_roll"
    if roll_col not in master.columns:
        continue
    r  = master[col].abs()
    rl = master[roll_col] == 1
    print(f"  {ASSET_LABELS.get(col, col):22s} roll days: {r[rl].mean()*100:.4f}%  "
          f"({rl.sum():>4} obs)   other days: {r[~rl].mean()*100:.4f}%  "
          f"ratio {r[rl].mean()/r[~rl].mean():.2f}x")

print("\n=== Realised tenor of each constant-maturity series ===")
for col, target in [("ntnb", 5.0), ("ltn", 2.0), ("ntnf", 10.0), ("lft", 1.0)]:
    t = master[f"{col}_ttm"].dropna()
    print(f"  {ASSET_LABELS.get(col, col):22s} target {target:4.1f}y   "
          f"realised mean {t.mean():5.2f}y   "
          f"mean |gap| {np.abs(t-target).mean():.2f}y   "
          f"worst {np.abs(t-target).max():.2f}y")

## 6. Summary statistics table

Key stats per asset across full sample — this becomes Table A1 in the whitepaper appendix.

In [ ]:
from scipy import stats as scipy_stats

ret_cols = ["ibov", "ntnb", "ltn", "ntnf", "lft"]
df = master[ret_cols].dropna(how="all") * 100  # in percent

rows = []
for col in ret_cols:
    s = df[col].dropna()
    # Annualise assuming 252 trading days
    ann_ret = s.mean() * 252
    ann_vol = s.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol  # no risk-free deduction (use CDI-adjusted later)

    # Max drawdown
    px = np.exp(s.cumsum() / 100)
    dd = (px / px.cummax() - 1).min() * 100

    rows.append({
        "Asset":        ASSET_LABELS[col],
        "Obs":          len(s),
        "Ann. Return%": round(ann_ret, 2),
        "Ann. Vol%":    round(ann_vol, 2),
        "Sharpe":       round(sharpe, 3),
        "Skewness":     round(float(scipy_stats.skew(s)), 3),
        "Kurtosis":     round(float(scipy_stats.kurtosis(s)), 3),
        "Max DD%":      round(dd, 2),
    })

summary = pd.DataFrame(rows).set_index("Asset")
print("=== Full-sample summary statistics (log returns, daily) ===")
print(summary.to_string())
summary.to_csv("../outputs/nb_tbl_summary_stats.csv")
print("\nSaved: outputs/nb_tbl_summary_stats.csv")

## 7. Data quality report

Identify gaps, extreme values, and suspicious observations to flag in the methodology section.

In [ ]:
ret_cols = ["ibov", "ntnb", "ltn", "ntnf", "lft"]
df = master[ret_cols] * 100  # pct

print("=== Extreme daily moves (|return| > 5%) ===")
for col in ret_cols:
    s = df[col].dropna()
    extremes = s[s.abs() > 5].sort_values()
    if len(extremes):
        print(f"\n{ASSET_LABELS[col]}:")
        for dt, val in extremes.items():
            print(f"  {dt.date()}  {val:+.2f}%")
    else:
        print(f"\n{ASSET_LABELS[col]}: no extreme moves")

print("\n=== NaN gaps by year ===")
nan_by_year = (master[ret_cols].isnull()
                .groupby(master.index.year).sum()
                .rename(columns=ASSET_LABELS))
print(nan_by_year[nan_by_year.sum(axis=1) > 0].to_string())

## ✅ Notebook 01 complete

**What we have:**
- `data/processed/master_returns.csv` — ~5,600 rows, 2004–2026 (bonds from 2005-01-03)
- Asset return series: Ibovespa, NTN-B 5y, LTN 2y, NTN-F 10y, LFT 1y, LFT long, BRL/USD
- Macro levels: EMBI+ Brazil (bps), CDI (% p.a.), Selic, IPCA, BRL/USD
- Per-bond diagnostics: realised tenor, quoted yield, roll flag
- Event labels: `crisis`, `regime`

**What the validation actually showed** — read the printed output above rather than
this cell; the numbers are computed, not asserted. The checks that must pass:

| Check | Pass condition |
|-------|----------------|
| LFT vs compounded CDI | rho > 0.9, annualised gap < 1pp |
| Roll-day returns | not materially larger than non-roll days |
| Realised tenor | close to the 5y / 2y / 10y / 1y targets |
| EMBI level | a spread in basis points (median ~250), not an FX rate |
| CDI level | 2–20% p.a. — a plausible Brazilian policy rate |

`src/fetch.py::validate_master` runs the range and coverage checks automatically on
every rebuild and prints a pass/fail line per series.

**Next:** `02_descriptive.ipynb` — regime-split statistics and unconditional correlation matrices